# Indian TTS — Staged Training with Podcast Validation

Train Indian English TTS (male & female) step by step, with a **fixed podcast script** generated at every stage so you can hear the improvement.

| Stage | Time | Cumulative | Compute Units | Podcast Sounds Like |
|-------|------|-----------|--------------|--------------------|
| **0** | ~2 min | 2 min | ~0.5 CU | Static/noise |
| **1** | ~30 min | 32 min | ~7 CU | Buzzy, some energy |
| **2** | ~30 min | 1 hr | ~14 CU | Speech-like noise |
| **3** | ~1 hr | 2 hrs | ~28 CU | Vowel sounds, patterns |
| **4** | ~6 hrs | 8 hrs | ~113 CU | Partially intelligible words |
| **5** | ~20 hrs | 28 hrs | ~393 CU | Clear Indian English |

**The same AI podcast (Arjun & Priya discussing AI in India) is generated at every stage.**

**Requirements:** Colab Pro+ ($49.99/month, 500 CU) with A100 GPU

---
## Step 1: Check GPU

Go to **Runtime > Change runtime type > A100 GPU** before running.

In [ ]:
import torch
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
assert torch.cuda.is_available(), "No GPU! Go to Runtime > Change runtime type > A100"
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.0f} GB")

## Step 2: Login to HuggingFace

Required to download the Svarah dataset. Get a token at https://huggingface.co/settings/tokens

**Also:** Go to https://huggingface.co/datasets/ai4bharat/Svarah and click **"Agree and access repository"** if prompted.

In [ ]:
from huggingface_hub import notebook_login, HfApi

# Check if already logged in
try:
    api = HfApi()
    user = api.whoami()
    print(f"Already logged in as: {user['name']}")
except Exception:
    print("Not logged in. Please enter your HuggingFace token below:")
    print("(Get one at https://huggingface.co/settings/tokens)")
    notebook_login()

## Step 3: Clone repo and install dependencies

In [ ]:
import os

# Always start from a clean state
os.chdir('/content')

# Remove old clone if broken
if os.path.exists('/content/indian_tts') and not os.path.exists('/content/indian_tts/.git'):
    !rm -rf /content/indian_tts

# Clone fresh if needed
if not os.path.exists('/content/indian_tts'):
    !git clone https://github.com/seetha0712/text2speech_1.git /content/indian_tts

# Verify clone worked
assert os.path.isdir('/content/indian_tts/.git'), "Clone failed! Is the repo public?"

# Switch to repo and get latest code
os.chdir('/content/indian_tts')
!git checkout claude/custom-indian-tts-model-TUAjJ
!git pull origin claude/custom-indian-tts-model-TUAjJ

# Install dependencies
!pip install -q -r requirements.txt
!pip install -q -e .
!apt-get install -qq espeak-ng > /dev/null 2>&1

print(f"\nWorking directory: {os.getcwd()}")
print("Setup complete!")

## Step 4: Download Indian English data

Downloads only legally safe datasets:
- **Svarah** (CC-BY 4.0) — 9.6 hrs Indian English, 117 speakers, male & female
- **Common Voice** (CC-0) — bonus data if community mirror is available

In [ ]:
os.chdir('/content/indian_tts')
!python -m indian_tts.data.preprocess \
    --source all \
    --output /content/data \
    --max-hours 15 \
    --min-upvotes 2

## Step 5: Create training config

In [ ]:
import yaml

with open('configs/colab_a100_config.yaml') as f:
    config = yaml.safe_load(f)

config['data']['training_files'] = '/content/data/train.txt'
config['data']['validation_files'] = '/content/data/val.txt'
config['paths']['output_dir'] = '/content/outputs'
config['paths']['checkpoint_dir'] = '/content/outputs/checkpoints'
config['paths']['log_dir'] = '/content/outputs/logs'

with open('/content/stage_config.yaml', 'w') as f:
    yaml.dump(config, f, default_flow_style=False)

print("Config ready!")
print(f"Batch size: {config['training']['batch_size']}")

# Verify data
for fname in ['train.txt', 'val.txt']:
    path = f'/content/data/{fname}'
    if os.path.exists(path):
        with open(path) as fh:
            lines = [l for l in fh if l.strip() and not l.startswith('#')]
        print(f"{fname}: {len(lines)} samples")
    else:
        print(f"ERROR: {path} not found! Re-run the data download cell.")

## Step 6: Setup helpers

Helper functions to listen to podcasts and compare across stages.

In [ ]:
import IPython.display as ipd
import glob

def listen_podcast(stage_num):
    """Play the full podcast and individual lines from a given stage."""
    podcast_dir = f'/content/outputs/podcast_stage_{stage_num}'
    full_path = f'{podcast_dir}/podcast_full.wav'

    if not os.path.exists(full_path):
        print(f"No podcast found for stage {stage_num}.")
        return

    print(f"{'=' * 60}")
    print(f"  PODCAST - Stage {stage_num}")
    print(f"  Arjun (male) & Priya (female) discuss AI in India")
    print(f"{'=' * 60}")
    print(f"\nFull podcast:")
    ipd.display(ipd.Audio(full_path))

    lines = sorted(glob.glob(f'{podcast_dir}/line_*.wav'))[:4]
    script_lines = [
        ('Priya', 'Welcome to AI India, the podcast where we explore...'),
        ('Arjun', 'And I am Arjun. Today we are talking about...'),
        ('Priya', 'India now has over three hundred AI startups...'),
        ('Arjun', 'Many of these companies are solving uniquely Indian problems...'),
    ]
    if lines:
        print(f"\nFirst few lines (to hear each speaker):")
        for wav_path, (speaker, preview) in zip(lines, script_lines):
            print(f"\n  [{speaker}] {preview}")
            ipd.display(ipd.Audio(wav_path))

def compare_podcasts():
    """Play all available stage podcasts side by side."""
    print("COMPARE: How the podcast improves across stages\n")
    for stage in range(6):
        full_path = f'/content/outputs/podcast_stage_{stage}/podcast_full.wav'
        if os.path.exists(full_path):
            import soundfile as sf
            data, sr = sf.read(full_path)
            duration = len(data) / sr
            print(f"\nStage {stage} ({duration:.0f}s):")
            ipd.display(ipd.Audio(full_path))

print("Helpers ready: listen_podcast(stage_num), compare_podcasts()")

---
# Training Stages

Run each stage one at a time. Check the PASS/FAIL result and listen to the podcast before proceeding.

---
## Stage 0 - Sanity Check (~2 min)

Does it run without crashing? Podcast will be **pure noise** - that's expected.

In [ ]:
os.chdir('/content/indian_tts')
!python -m indian_tts.validate --config /content/stage_config.yaml --stage 0

In [ ]:
listen_podcast(0)  # Expect: noise/static

**PASSED?** Proceed to Stage 1. **FAILED?** Check errors above (OOM -> reduce batch_size to 32 in Step 5).

---
## Stage 1 - Smoke Test (~30 min)

Are losses decreasing? Podcast will be **buzzy/noisy but not silent**.

In [ ]:
os.chdir('/content/indian_tts')
ckpts = sorted(glob.glob('/content/outputs/checkpoints/checkpoint_*.pt'))
resume = ckpts[-1] if ckpts else ''
print(f"Resuming from: {resume}")
!python -m indian_tts.validate --config /content/stage_config.yaml --stage 1 --resume {resume}

In [ ]:
listen_podcast(1)  # Expect: buzzy, some energy, not pure static

**PASSED?** Proceed to Stage 2. **FAILED?** Lower learning_rate to 0.0001 in Step 5.

---
## Stage 2 - Early Signal (~1 hr cumulative)

Is audio structure emerging? Podcast will have **speech-like noise**.

In [ ]:
os.chdir('/content/indian_tts')
ckpts = sorted(glob.glob('/content/outputs/checkpoints/checkpoint_*.pt'))
resume = ckpts[-1] if ckpts else ''
!python -m indian_tts.validate --config /content/stage_config.yaml --stage 2 --resume {resume}

In [ ]:
listen_podcast(2)
print("\n--- Compare stages 0 vs 1 vs 2: ---")
compare_podcasts()

**PASSED + male/female differ?** Proceed to Stage 3.

---
## Stage 3 - Quality Gate (~2 hrs cumulative) - THE GO/NO-GO POINT

Do different texts produce different outputs? Are Arjun and Priya distinguishable?

**If this passes, the 6-hour investment for Stage 4 is worth it.**

In [ ]:
os.chdir('/content/indian_tts')
ckpts = sorted(glob.glob('/content/outputs/checkpoints/checkpoint_*.pt'))
resume = ckpts[-1] if ckpts else ''
!python -m indian_tts.validate --config /content/stage_config.yaml --stage 3 --resume {resume}

In [ ]:
listen_podcast(3)
print("\n--- Compare all stages so far: ---")
compare_podcasts()

### DECISION POINT

**PASSED?** The model IS learning. Stage 4 (6 hrs) will produce intelligible speech. **Go for it.**

**FAILED?** STOP. Do not spend 6 more hours. Re-download with more data or adjust config.

---
## Stage 4 - Full Training (~8 hrs cumulative)

First intelligible speech. **Back up to Google Drive first!**

In [ ]:
# Back up checkpoints to Google Drive
from google.colab import drive
import shutil
drive.mount('/content/drive')
drive_backup = '/content/drive/MyDrive/indian_tts_checkpoints'
os.makedirs(drive_backup, exist_ok=True)
for ckpt in sorted(glob.glob('/content/outputs/checkpoints/checkpoint_*.pt'))[-2:]:
    shutil.copy2(ckpt, drive_backup)
    print(f"Backed up: {os.path.basename(ckpt)}")
shutil.copy2('/content/stage_config.yaml', drive_backup)
print("Backup done!")

In [ ]:
%load_ext tensorboard
%tensorboard --logdir /content/outputs/logs

In [ ]:
os.chdir('/content/indian_tts')
ckpts = sorted(glob.glob('/content/outputs/checkpoints/checkpoint_*.pt'))
resume = ckpts[-1] if ckpts else ''
print(f"Resuming from: {resume}")
print("This will take ~6 hours...")
!python -m indian_tts.validate --config /content/stage_config.yaml --stage 4 --resume {resume}

In [ ]:
listen_podcast(4)
print("\n--- Full comparison across all stages: ---")
compare_podcasts()

# Backup to Drive
import shutil
drive_backup = '/content/drive/MyDrive/indian_tts_checkpoints'
for ckpt in sorted(glob.glob('/content/outputs/checkpoints/checkpoint_*.pt'))[-2:]:
    shutil.copy2(ckpt, drive_backup)
for stage in range(5):
    src = f'/content/outputs/podcast_stage_{stage}/podcast_full.wav'
    if os.path.exists(src):
        shutil.copy2(src, os.path.join(drive_backup, f'podcast_stage_{stage}.wav'))
print("All backed up to Google Drive!")

---
## Stage 5 - Extended Training (~28 hrs cumulative, optional)

For highest quality. May need multiple Colab sessions - resume from Drive.

In [ ]:
# Restore from Drive if this is a new session
import glob, os, shutil
drive_backup = '/content/drive/MyDrive/indian_tts_checkpoints'
local_ckpts = sorted(glob.glob('/content/outputs/checkpoints/checkpoint_*.pt'))
drive_ckpts = sorted(glob.glob(f'{drive_backup}/checkpoint_*.pt'))
if not local_ckpts and drive_ckpts:
    os.makedirs('/content/outputs/checkpoints', exist_ok=True)
    shutil.copy2(drive_ckpts[-1], '/content/outputs/checkpoints/')
    print(f"Restored: {os.path.basename(drive_ckpts[-1])}")

os.chdir('/content/indian_tts')
ckpts = sorted(glob.glob('/content/outputs/checkpoints/checkpoint_*.pt'))
resume = ckpts[-1] if ckpts else ''
!python -m indian_tts.validate --config /content/stage_config.yaml --stage 5 --resume {resume}

In [ ]:
listen_podcast(5)
print("\n--- FINAL COMPARISON: ---")
compare_podcasts()

---
## Generate Custom Speech

Use the model on any text you want.

In [ ]:
from indian_tts.inference import IndianTTS
import IPython.display as ipd

ckpts = sorted(glob.glob('/content/outputs/checkpoints/checkpoint_*.pt'))
tts = IndianTTS(ckpts[-1])

# Try your own text!
my_text = "Hello, this is my custom Indian text to speech model speaking."

for voice in ['male', 'female']:
    audio = tts.synthesize(my_text, voice=voice)
    print(f"\n[{voice.upper()}] {my_text}")
    ipd.display(ipd.Audio(audio, rate=tts.sampling_rate))

In [ ]:
# Generate a fresh podcast with different settings
ckpts = sorted(glob.glob('/content/outputs/checkpoints/checkpoint_*.pt'))
!python -m indian_tts.podcast_demo \
    --checkpoint {ckpts[-1]} \
    --output /content/outputs/podcast_custom \
    --speed 1.0 \
    --expressiveness 0.8

ipd.display(ipd.Audio('/content/outputs/podcast_custom/podcast_full.wav'))